# Logs

> Generating / formatting the base logging instance

In [ ]:
#| default_exp mcp

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
from __future__ import annotations
from typing import Any, Dict, List, Optional, Annotated
from enum import Enum

import os, sys, json
from pathlib import Path

import typer
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich import print as rprint

from mcp.server.fastmcp import FastMCP

from nbdev_mcp import __version__
from nbdev_mcp.utils.logs import log
from nbdev_mcp.utils.config import CURRENT_PROJECT
from nbdev_mcp.utils.paths import (
    resolve_selector,
    get_vscode_mcp_path,
    get_vscode_settings_path as get_vscode_settings_path_from_utils,
    get_cursor_mcp_path,
    get_cursor_settings_path as get_cursor_settings_path_from_utils,
    get_provider_config_paths,
    select_provider_config_path,
    get_claude_config_path as get_claude_code_config_path,
    get_codex_config_path as get_codex_toml_config_path,
)
from nbdev_mcp.utils.subprocess import watch_notebooks
from nbdev_mcp.resources import add_resources
from nbdev_mcp.tools import (
    add_project_tools, add_env_tools, add_nbdev_tools, 
    add_notebook_editing_tools, add_lint_tools, 
    add_analysis_tools, add_tests_tools
)
from nbdev_mcp.prompts import add_prompts
from nbdev_mcp.tasks import add_task_tools, enable_nbdev_tasks
from nbdev_mcp.tests.agent_e2e import add_recording_tools, enable_auto_recording

console = Console();  # semicolon suppresses notebook output


## HTTP Utils

In [ ]:
#| export
def set_http_path_if_supported(target_path: str) -> bool:
    """Try to set the HTTP mount path on the MCP SDK settings.
    
    Parameters
    ----------
    target_path : str
        The path to set (e.g., '/mcp').
    
    Returns
    -------
    bool
        True if successfully set, False if not supported.
    """
    import mcp
    try:
        mcp.settings.streamable_http_path = target_path  # type: ignore[attr-defined]
        return True
    except Exception:
        try:
            mcp.settings.http_path = target_path  # type: ignore[attr-defined]
            return True
        except Exception:
            return False

## Create the MCP

In [ ]:
#| export
def create_nbdev_mcp(name: str = "mcp.nbdev") -> FastMCP:
    """Create and configure the nbdev MCP server with all resources, tools, and prompts."""
    mcp = FastMCP(name)
    
    # Enable experimental tasks API
    enable_nbdev_tasks(mcp)
    
    # Attach all nbdev-related resources, tools, and prompts
    add_resources(mcp)
    add_project_tools(mcp)
    add_env_tools(mcp)
    add_nbdev_tools(mcp)
    add_notebook_editing_tools(mcp)  # CRITICAL: Notebook editing and workflow tools
    add_prompts(mcp)  # Philosophy prompts must come after tools are registered
    
    # Extensions: linting, analysis, and code generation tools
    add_lint_tools(mcp)
    add_analysis_tools(mcp)
    add_tests_tools(mcp)
    
    # Task-based tools for auditing, deduplication, and refactoring
    add_task_tools(mcp)
    
    # E2E testing: recording tools for agent behavior analysis
    add_recording_tools(mcp)
    enable_auto_recording(mcp)  # Enable automatic recording when recording is active
    
    return mcp

## `main`

In [ ]:
#| export
class Transport(str, Enum):
    """MCP transport modes."""
    stdio = "stdio"
    http = "http"
    streamable_http = "streamable-http"


class Provider(str, Enum):
    """Supported MCP client providers."""
    claude = "claude"
    vscode = "vscode"
    cursor = "cursor"
    codex = "codex"


app = typer.Typer(
    name="nbdev-mcp",
    help="[bold blue]nbdev MCP server[/] - Model Context Protocol for nbdev projects",
    rich_markup_mode="rich",
    no_args_is_help=False,
    add_completion=True,
    context_settings={"help_option_names": ["-h", "--help"]},
);  # semicolon suppresses notebook output


def version_callback(value: bool):
    if value:
        rprint(f"[bold blue]nbdev-mcp[/] [dim]v{__version__}[/]")
        raise typer.Exit()


@app.callback(invoke_without_command=True)
def callback(
    ctx: typer.Context,
    version: Annotated[bool, typer.Option(
        "-V", "--version", callback=version_callback, is_eager=True,
        help="Show version and exit."
    )] = False,
    project: Annotated[Optional[str], typer.Option(
        "-p", "--project", help="Path or alias for an nbdev project."
    )] = None,
    transport: Annotated[Transport, typer.Option(
        "-t", "--transport", help="Transport mode.",
        envvar="NBDEV_MCP_TRANSPORT"
    )] = Transport.stdio,
    host: Annotated[str, typer.Option(
        "-H", "--host", help="Host for HTTP transport.",
        envvar="NBDEV_MCP_HOST"
    )] = "127.0.0.1",
    port: Annotated[int, typer.Option(
        "-P", "--port", help="Port for HTTP transport.",
        envvar="NBDEV_MCP_PORT"
    )] = 8000,
    path: Annotated[str, typer.Option(
        "--path", help="URL path for HTTP.",
        envvar="NBDEV_MCP_PATH"
    )] = "/mcp",
    verbose: Annotated[bool, typer.Option(
        "-v", "--verbose", help="Enable debug output."
    )] = False,
    watch: Annotated[bool, typer.Option(
        "-w", "--watch", help="Watch notebooks and auto-export."
    )] = False,
    watch_interval: Annotated[float, typer.Option(
        "--watch-interval", help="Watch polling interval (seconds)."
    )] = 2.0,
    watch_cmd: Annotated[str, typer.Option(
        "--watch-cmd", help="Command on change."
    )] = "nbdev_export",
):
    """Run the MCP server (default command)."""
    if ctx.invoked_subcommand is None:
        _run_server(
            project=project,
            transport=transport,
            host=host,
            port=port,
            path=path,
            verbose=verbose,
            watch=watch,
            watch_interval=watch_interval,
            watch_cmd=watch_cmd,
        )


## Next

In [ ]:
#| export


## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()

## Install helpers

In [ ]:
#| export
def get_claude_config_path() -> Path:
    """Get active Claude config path (Code or Desktop)."""
    selected = select_provider_config_path("claude")
    return selected if selected is not None else get_claude_code_config_path()


def get_vscode_config_path() -> Path:
    """Get VS Code MCP config path (mcp.json)."""
    return get_vscode_mcp_path()


def get_vscode_settings_path() -> Path:
    """Get VS Code settings.json path."""
    return get_vscode_settings_path_from_utils()


def get_cursor_config_path() -> Path:
    """Get Cursor MCP config path (mcp.json)."""
    return get_cursor_mcp_path()


def get_cursor_settings_path() -> Path:
    """Get Cursor settings.json path."""
    return get_cursor_settings_path_from_utils()


def get_codex_config_path() -> Path:
    """Get active Codex config path (prefers TOML, supports legacy JSON)."""
    selected = select_provider_config_path("codex")
    return selected if selected is not None else get_codex_toml_config_path()


def get_config_paths(provider: Provider) -> List[Path]:
    """Get candidate config paths for a provider in priority order."""
    return get_provider_config_paths(provider.value)


def get_config_path(provider: Provider) -> Path:
    """Get selected config path for a provider."""
    selected = select_provider_config_path(provider.value)
    if selected is None:
        raise ValueError(f"Unsupported provider: {provider.value}")
    return selected


def get_config_format(config_path: Path) -> str:
    """Get config format from file extension."""
    return "toml" if config_path.suffix.lower() == ".toml" else "json"


def get_mcp_key(provider: Provider, config_format: Optional[str] = None) -> str:
    """Get the MCP servers key for a provider config."""
    if config_format == "toml":
        return "mcp_servers"
    match provider:
        case Provider.claude | Provider.codex:
            return "mcpServers"
        case Provider.vscode | Provider.cursor:
            return "servers"


def get_python_path() -> str:
    """Get current Python interpreter path."""
    return sys.executable


def make_server_config() -> dict:
    """Generate server configuration dict."""
    return {
        "command": get_python_path(),
        "args": ["-u", "-m", "nbdev_mcp"],
    }


def make_server_config_for_provider(provider: Provider, auto_start: bool = False) -> dict:
    """Generate server configuration dict for a specific provider."""
    base = {
        "command": get_python_path(),
        "args": ["-u", "-m", "nbdev_mcp"],
    }
    # VS Code/Cursor mcp.json format includes "type" and optional "autoStart"
    if provider in (Provider.vscode, Provider.cursor):
        config = {"type": "stdio", **base}
        if auto_start:
            config["autoStart"] = True
        return config
    return base


def get_wrapper_script_path() -> Path:
    """Get path for the keep-alive wrapper script."""
    if sys.platform == "win32":
        return Path.home() / ".nbdev-mcp" / "nbdev-mcp-wrapper.bat"
    return Path.home() / ".nbdev-mcp" / "nbdev-mcp-wrapper.sh"


def generate_wrapper_script() -> str:
    """Generate a keep-alive wrapper script that monitors and restarts the MCP."""
    python_path = get_python_path()
    if sys.platform == "win32":
        return f'''@echo off
REM nbdev-mcp keep-alive wrapper
:loop
"{python_path}" -u -m nbdev_mcp %*
echo MCP exited with code %ERRORLEVEL%, restarting in 2 seconds...
timeout /t 2 /nobreak >nul
goto loop
'''
    return f'''#!/usr/bin/env bash
# nbdev-mcp keep-alive wrapper
# Monitors and restarts the MCP server if it exits

PYTHON="{python_path}"
RESTART_DELAY=2

while true; do
    echo "[nbdev-mcp] Starting server..."
    "$PYTHON" -u -m nbdev_mcp "$@"
    EXIT_CODE=$?
    echo "[nbdev-mcp] Server exited with code $EXIT_CODE, restarting in ${{RESTART_DELAY}}s..."
    sleep $RESTART_DELAY
done
'''


def install_wrapper_script(dry_run: bool = False) -> Path:
    """Install the keep-alive wrapper script."""
    script_path = get_wrapper_script_path()
    script_content = generate_wrapper_script()

    if dry_run:
        console.print(f"\n[bold cyan]Wrapper Script[/]")
        console.print(f"[dim]File:[/] {script_path}")
        console.print(f"\n[dim]Script content:[/]")
        console.print(script_content)
        return script_path

    script_path.parent.mkdir(parents=True, exist_ok=True)
    script_path.write_text(script_content)
    if sys.platform != "win32":
        script_path.chmod(0o755)
    console.print(f"[green]✓[/] Wrapper script installed: {script_path}")
    return script_path


def parse_jsonc(text: str) -> dict:
    """Parse JSON with comments and trailing commas support."""
    import re

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    normalized = re.sub(r'//[^\n]*', '', text)
    normalized = re.sub(r'/\*.*?\*/', '', normalized, flags=re.DOTALL)
    normalized = re.sub(r',(\s*[}\]])', r'\1', normalized)

    try:
        return json.loads(normalized)
    except json.JSONDecodeError as exc:
        raise ValueError(f"Invalid JSON/JSONC: {exc}") from exc


def parse_toml(text: str) -> dict:
    """Parse TOML and raise ValueError on failure."""
    import tomllib

    try:
        return tomllib.loads(text)
    except Exception as exc:
        raise ValueError(f"Invalid TOML: {exc}") from exc


def render_diff(before: str, after: str, path: Path) -> str:
    """Render unified diff between two text blobs."""
    from difflib import unified_diff

    before_lines = before.splitlines(keepends=True)
    after_lines = after.splitlines(keepends=True)
    diff_lines = list(unified_diff(before_lines, after_lines, fromfile=f"{path} (before)", tofile=f"{path} (after)"))
    return "".join(diff_lines)


def make_backup(path: Path) -> Optional[Path]:
    """Create timestamped backup for an existing config file."""
    if not path.exists():
        return None
    from datetime import datetime

    stamp = datetime.now().strftime("%Y%m%d%H%M%S")
    backup_path = path.with_name(f"{path.name}.{stamp}.bak")
    backup_path.write_text(path.read_text())
    return backup_path


def write_text_config(path: Path, content: str, backup: bool = True) -> Optional[Path]:
    """Write config text with optional backup of existing file."""
    path.parent.mkdir(parents=True, exist_ok=True)
    backup_path = make_backup(path) if backup else None
    path.write_text(content)
    return backup_path


def reconcile_server_config(
    provider: Provider,
    current_server: Optional[dict],
    strategy: str,
    auto_start: bool,
) -> dict:
    """Reconcile current and desired server configs."""
    desired = make_server_config_for_provider(provider, auto_start=auto_start)
    if strategy == "replace":
        return desired

    merged = dict(current_server or {})
    merged.update(desired)
    if provider in (Provider.vscode, Provider.cursor) and auto_start:
        merged["autoStart"] = True
    return merged


def toml_value(value: Any) -> str:
    """Render a Python value as TOML literal."""
    if isinstance(value, bool):
        return "true" if value else "false"
    if isinstance(value, (int, float)):
        return str(value)
    if isinstance(value, str):
        return json.dumps(value)
    if isinstance(value, list):
        inner = ", ".join(toml_value(item) for item in value)
        return f"[{inner}]"
    if isinstance(value, dict):
        parts = [f"{key} = {toml_value(val)}" for key, val in value.items()]
        return "{ " + ", ".join(parts) + " }"
    return json.dumps(str(value))


def render_codex_toml_block(server_config: dict) -> str:
    """Render the [mcp_servers.nbdev] table for Codex TOML config."""
    ordered_keys: List[str] = []
    for key in ["command", "args", "env", "cwd"]:
        if key in server_config:
            ordered_keys.append(key)
    for key in sorted(server_config.keys()):
        if key not in ordered_keys:
            ordered_keys.append(key)

    lines = ["[mcp_servers.nbdev]"]
    for key in ordered_keys:
        lines.append(f"{key} = {toml_value(server_config[key])}")
    return "\n".join(lines) + "\n"


def upsert_toml_table(text: str, table_name: str, table_block: str) -> tuple[str, bool]:
    """Insert or replace a TOML table block by table name."""
    import re

    normalized = text if text.endswith("\n") or text == "" else text + "\n"
    pattern = re.compile(rf"(?ms)^\[{re.escape(table_name)}\]\n(?:.*\n)*?(?=^\[|\Z)")
    replacement = table_block if table_block.endswith("\n") else table_block + "\n"

    if pattern.search(normalized):
        return pattern.sub(replacement, normalized, count=1), True

    joiner = "\n" if normalized.strip() else ""
    return normalized + joiner + replacement, False


def remove_toml_table(text: str, table_name: str) -> tuple[str, bool]:
    """Remove a TOML table block by table name."""
    import re

    normalized = text if text.endswith("\n") or text == "" else text + "\n"
    pattern = re.compile(rf"(?ms)^\[{re.escape(table_name)}\]\n(?:.*\n)*?(?=^\[|\Z)")
    new_text = pattern.sub("", normalized, count=1)
    changed = new_text != normalized

    while "\n\n\n" in new_text:
        new_text = new_text.replace("\n\n\n", "\n\n")
    if new_text and not new_text.endswith("\n"):
        new_text += "\n"
    return new_text, changed


def enable_mcp_autostart_in_settings(provider: Provider, dry_run: bool = False) -> bool:
    """Enable chat.mcp.autostart in VS Code/Cursor settings.json."""
    if provider == Provider.vscode:
        settings_path = get_vscode_settings_path()
    elif provider == Provider.cursor:
        settings_path = get_cursor_settings_path()
    else:
        return False

    if settings_path.exists():
        try:
            settings = parse_jsonc(settings_path.read_text())
        except ValueError as exc:
            console.print(f"[red]✗[/] {provider.value}: {exc}")
            return False
    else:
        settings = {}

    if settings.get("chat.mcp.autostart") is True:
        console.print("[dim]  chat.mcp.autostart already enabled[/]")
        return True

    settings["chat.mcp.autostart"] = True
    after_text = json.dumps(settings, indent=2) + "\n"

    if dry_run:
        console.print(f"\n[bold cyan]{provider.value.title()} settings[/]")
        console.print(f"[dim]File:[/] {settings_path}")
        console.print("[dim]Action:[/] Set chat.mcp.autostart = true")
        return True

    settings_path.parent.mkdir(parents=True, exist_ok=True)
    settings_path.write_text(after_text)
    console.print(f"[green]✓[/] Enabled chat.mcp.autostart in {settings_path}")
    return True


def update_provider_config(
    provider: Provider,
    dry_run: bool = False,
    auto_start: bool = False,
    strategy: str = "merge",
    backup: bool = True,
) -> bool:
    """Install/update nbdev-mcp entry for a provider using reconcile strategy."""
    strategy_value = strategy.lower().strip()
    if strategy_value not in {"merge", "replace"}:
        console.print(f"[red]✗[/] {provider.value}: invalid strategy '{strategy}' (use merge or replace)")
        return False

    config_paths = get_config_paths(provider)
    config_path = get_config_path(provider)
    config_format = get_config_format(config_path)
    mcp_key = get_mcp_key(provider, config_format=config_format)

    before_text = config_path.read_text() if config_path.exists() else ""
    existed = config_path.exists()

    try:
        if config_format == "toml":
            current_server: Optional[dict] = None
            if existed:
                parsed = parse_toml(before_text)
                current_server = parsed.get("mcp_servers", {}).get("nbdev")

            server_config = reconcile_server_config(provider, current_server, strategy_value, auto_start)
            block = render_codex_toml_block(server_config)
            after_text, replaced = upsert_toml_table(before_text, "mcp_servers.nbdev", block)
            action = "Update existing config" if replaced else ("Update existing config" if existed else "Create new config file")
        else:
            if existed:
                config = parse_jsonc(before_text)
            else:
                config = {}

            if mcp_key not in config or not isinstance(config[mcp_key], dict):
                config[mcp_key] = {}

            current_server = config[mcp_key].get("nbdev") if isinstance(config[mcp_key], dict) else None
            config[mcp_key]["nbdev"] = reconcile_server_config(provider, current_server, strategy_value, auto_start)
            after_text = json.dumps(config, indent=2) + "\n"
            action = "Update existing config" if existed else "Create new config file"
    except ValueError as exc:
        console.print(f"[red]✗[/] {provider.value}: {exc}")
        console.print("[dim]No config changes were written.[/]")
        return False

    diff = render_diff(before_text, after_text, config_path)

    if dry_run:
        console.print(f"\n[bold cyan]{provider.value.title()}[/]")
        console.print(f"[dim]File:[/] {config_path}")
        console.print(f"[dim]Format:[/] {config_format}")
        console.print(f"[dim]Candidates:[/] {', '.join(str(p) for p in config_paths)}")
        console.print(f"[dim]Action:[/] {action}")
        console.print(f"[dim]Strategy:[/] {strategy_value}")
        if diff:
            console.print("\n[dim]Unified diff:[/]")
            console.print(diff)
        else:
            console.print("\n[dim]No changes (already up to date).[/]")
    else:
        if before_text == after_text:
            console.print(f"[green]✓[/] {provider.value}: already up to date ({config_path})")
        else:
            backup_path = write_text_config(config_path, after_text, backup=backup)
            console.print(f"[green]✓[/] Updated [bold]{provider.value}[/]: {config_path}")
            if backup_path is not None:
                console.print(f"[dim]Backup:[/] {backup_path}")

    if auto_start and provider in (Provider.vscode, Provider.cursor):
        enable_mcp_autostart_in_settings(provider, dry_run=dry_run)

    return True


def install_to_provider(
    provider: Provider,
    dry_run: bool = False,
    auto_start: bool = False,
    backup: bool = True,
) -> bool:
    """Install nbdev-mcp to a provider config."""
    return update_provider_config(
        provider,
        dry_run=dry_run,
        auto_start=auto_start,
        strategy="replace",
        backup=backup,
    )


def uninstall_from_provider(provider: Provider, dry_run: bool = False, backup: bool = True) -> bool:
    """Remove nbdev-mcp from a provider config."""
    config_paths = get_config_paths(provider)
    config_path = get_config_path(provider)
    config_format = get_config_format(config_path)
    mcp_key = get_mcp_key(provider, config_format=config_format)

    if not config_path.exists():
        console.print(f"[yellow]⚠[/] {provider.value}: Config not found")
        return False

    before_text = config_path.read_text()

    try:
        if config_format == "toml":
            parse_toml(before_text)
            after_text, changed = remove_toml_table(before_text, "mcp_servers.nbdev")
            if not changed:
                console.print(f"[yellow]⚠[/] {provider.value}: nbdev-mcp not installed")
                return False
        else:
            config = parse_jsonc(before_text)
            if mcp_key not in config or "nbdev" not in config.get(mcp_key, {}):
                console.print(f"[yellow]⚠[/] {provider.value}: nbdev-mcp not installed")
                return False
            del config[mcp_key]["nbdev"]
            after_text = json.dumps(config, indent=2) + "\n"
    except ValueError as exc:
        console.print(f"[red]✗[/] {provider.value}: {exc}")
        console.print("[dim]No config changes were written.[/]")
        return False

    diff = render_diff(before_text, after_text, config_path)

    if dry_run:
        console.print(f"\n[bold cyan]{provider.value.title()}[/]")
        console.print(f"[dim]File:[/] {config_path}")
        console.print(f"[dim]Format:[/] {config_format}")
        console.print(f"[dim]Candidates:[/] {', '.join(str(p) for p in config_paths)}")
        console.print(f"[dim]Action:[/] Remove nbdev from {mcp_key}")
        if diff:
            console.print("\n[dim]Unified diff:[/]")
            console.print(diff)
        else:
            console.print("\n[dim]No changes.[/]")
        return True

    if before_text == after_text:
        console.print(f"[green]✓[/] {provider.value}: already removed")
        return True

    backup_path = write_text_config(config_path, after_text, backup=backup)
    console.print(f"[green]✓[/] Removed from [bold]{provider.value}[/]")
    if backup_path is not None:
        console.print(f"[dim]Backup:[/] {backup_path}")
    return True


def check_provider_status(provider: Provider) -> dict:
    """Check installation status for a provider."""
    config_paths = get_config_paths(provider)
    config_path = get_config_path(provider)
    config_format = get_config_format(config_path)
    mcp_key = get_mcp_key(provider, config_format=config_format)

    exists = config_path.exists()
    if not exists:
        return {
            "provider": provider.value,
            "installed": False,
            "exists": False,
            "path": str(config_path),
            "format": config_format,
            "candidates": [str(p) for p in config_paths],
        }

    try:
        if config_format == "toml":
            config = parse_toml(config_path.read_text())
            installed = isinstance(config.get("mcp_servers", {}).get("nbdev"), dict)
        else:
            config = parse_jsonc(config_path.read_text())
            installed = mcp_key in config and "nbdev" in config.get(mcp_key, {})
    except ValueError:
        return {
            "provider": provider.value,
            "installed": False,
            "exists": True,
            "path": str(config_path),
            "format": config_format,
            "candidates": [str(p) for p in config_paths],
            "error": "parse error",
        }

    autostart_enabled = None
    if provider in (Provider.vscode, Provider.cursor):
        settings_path = get_vscode_settings_path() if provider == Provider.vscode else get_cursor_settings_path()
        if settings_path.exists():
            try:
                settings = parse_jsonc(settings_path.read_text())
                autostart_enabled = settings.get("chat.mcp.autostart", False)
            except ValueError:
                pass

    result = {
        "provider": provider.value,
        "installed": installed,
        "exists": True,
        "path": str(config_path),
        "format": config_format,
        "candidates": [str(p) for p in config_paths],
    }
    if autostart_enabled is not None:
        result["autostart"] = autostart_enabled
    return result


In [ ]:
#| export
def _run_server(
    project: Optional[str] = None,
    transport: Transport = Transport.stdio,
    host: str = "127.0.0.1",
    port: int = 8000,
    path: str = "/mcp",
    verbose: bool = False,
    watch: bool = False,
    watch_interval: float = 2.0,
    watch_cmd: str = "nbdev_export",
) -> None:
    """Internal: run the MCP server."""
    if verbose:
        import logging
        logging.basicConfig(level=logging.DEBUG, format="%(message)s")
        log.setLevel(logging.DEBUG)
        console.print(f"[dim]nbdev-mcp v{__version__}[/]")

    # Set project from env or arg
    proj_path = None
    project = project or os.environ.get("NBDEV_MCP_PROJECT")
    if project:
        try:
            proj_path = resolve_selector(project)
            if verbose:
                console.print(f"[green]✓[/] Project: {proj_path}")
        except Exception as e:
            console.print(f"[red]✗[/] {e}")
            if watch:
                raise typer.Exit(1)

    # Watch mode
    if watch:
        if not proj_path:
            console.print("[red]✗[/] Watch mode requires --project")
            raise typer.Exit(1)
        watch_notebooks(proj_path, interval=watch_interval, on_change=watch_cmd)
        return

    # Build MCP
    mcp = create_nbdev_mcp()
    
    # Auto-recording for agent tests
    if os.environ.get("NBDEV_MCP_AUTO_RECORD"):
        from nbdev_mcp.tests.agent_e2e import start_recording
        start_recording()
        log.info("Auto-recording enabled via NBDEV_MCP_AUTO_RECORD")
    
    # Register shutdown handler to save session
    session_file = os.environ.get("NBDEV_MCP_SESSION_FILE")
    if session_file:
        import atexit
        def save_session_on_exit():
            from nbdev_mcp.tests.agent_e2e import stop_recording
            session = stop_recording()
            if session:
                session.save(Path(session_file))
                log.info(f"Session saved to {session_file}")
        atexit.register(save_session_on_exit)
    
    match transport:
        case Transport.stdio:
            mcp.run(transport="stdio")
        case Transport.streamable_http | Transport.http:
            default = (host == "127.0.0.1" and port == 8000 and path == "/mcp")
            if default and transport == Transport.streamable_http:
                mcp.run(transport="streamable-http")
            else:
                try:
                    import uvicorn
                except ImportError:
                    console.print("[red]✗[/] uvicorn required: [dim]pip install uvicorn[/]")
                    raise typer.Exit(1)
                if path != "/mcp":
                    set_http_path_if_supported(path)
                uvicorn.run(mcp.streamable_http_app(), host=host, port=port)


@app.command()
def run(
    project: Annotated[Optional[str], typer.Option(
        "-p", "--project", help="Path or alias for nbdev project."
    )] = None,
    transport: Annotated[Transport, typer.Option(
        "-t", "--transport", help="Transport: stdio, http, streamable-http.",
        envvar="NBDEV_MCP_TRANSPORT"
    )] = Transport.stdio,
    host: Annotated[str, typer.Option(
        "-H", "--host", help="Host for HTTP transport.",
        envvar="NBDEV_MCP_HOST"
    )] = "127.0.0.1",
    port: Annotated[int, typer.Option(
        "-P", "--port", help="Port for HTTP transport.",
        envvar="NBDEV_MCP_PORT"
    )] = 8000,
    path: Annotated[str, typer.Option(
        "--path", help="URL path for HTTP.",
        envvar="NBDEV_MCP_PATH"
    )] = "/mcp",
    verbose: Annotated[bool, typer.Option(
        "-v", "--verbose", help="Debug output."
    )] = False,
    watch: Annotated[bool, typer.Option(
        "-w", "--watch", help="Watch notebooks and auto-export."
    )] = False,
    watch_interval: Annotated[float, typer.Option(
        "--watch-interval", help="Watch polling interval (seconds)."
    )] = 2.0,
    watch_cmd: Annotated[str, typer.Option(
        "--watch-cmd", help="Command on change."
    )] = "nbdev_export",
):
    """Run the MCP server."""
    _run_server(
        project=project, transport=transport, host=host, port=port,
        path=path, verbose=verbose, watch=watch,
        watch_interval=watch_interval, watch_cmd=watch_cmd
    )

In [ ]:
#| export
@app.command()
def install(
    provider: Annotated[Optional[Provider], typer.Argument(
        help="Provider: claude, vscode, cursor, codex. Omit for all."
    )] = None,
    dry_run: Annotated[bool, typer.Option(
        "-d", "--dry-run", help="Show exact config changes without writing."
    )] = False,
    auto_start: Annotated[bool, typer.Option(
        "-a", "--auto-start", help="Auto-start server when VS Code/Cursor opens."
    )] = False,
    wrapper: Annotated[bool, typer.Option(
        "-w", "--wrapper", help="Install keep-alive wrapper script."
    )] = False,
    backup: Annotated[bool, typer.Option(
        "--backup/--no-backup", help="Backup config files before writing."
    )] = True,
):
    """Install nbdev-mcp to MCP client(s)."""
    providers = [provider] if provider else list(Provider)
    ok = True

    for p in providers:
        ok = install_to_provider(p, dry_run=dry_run, auto_start=auto_start, backup=backup) and ok

    if wrapper:
        install_wrapper_script(dry_run=dry_run)

    if not dry_run:
        console.print("\n[yellow]Restart your MCP client(s) to activate.[/]")

    if not ok:
        raise typer.Exit(1)


@app.command()
def update(
    provider: Annotated[Optional[Provider], typer.Argument(
        help="Provider: claude, vscode, cursor, codex. Omit for all."
    )] = None,
    dry_run: Annotated[bool, typer.Option(
        "-d", "--dry-run", help="Show exact config changes without writing."
    )] = False,
    auto_start: Annotated[bool, typer.Option(
        "-a", "--auto-start", help="Auto-start server when VS Code/Cursor opens."
    )] = False,
    strategy: Annotated[str, typer.Option(
        "-s", "--strategy", help="Reconcile strategy: merge or replace."
    )] = "merge",
    backup: Annotated[bool, typer.Option(
        "--backup/--no-backup", help="Backup config files before writing."
    )] = True,
):
    """Reconcile existing provider config with nbdev-mcp server config."""
    providers = [provider] if provider else list(Provider)
    ok = True

    for p in providers:
        ok = update_provider_config(
            p,
            dry_run=dry_run,
            auto_start=auto_start,
            strategy=strategy,
            backup=backup,
        ) and ok

    if not dry_run:
        console.print("\n[yellow]Restart your MCP client(s) to apply updates.[/]")

    if not ok:
        raise typer.Exit(1)


@app.command()
def uninstall(
    provider: Annotated[Optional[Provider], typer.Argument(
        help="Provider: claude, vscode, cursor, codex. Omit for all."
    )] = None,
    dry_run: Annotated[bool, typer.Option(
        "-d", "--dry-run", help="Show exact config changes without writing."
    )] = False,
    backup: Annotated[bool, typer.Option(
        "--backup/--no-backup", help="Backup config files before writing."
    )] = True,
):
    """Remove nbdev-mcp from MCP client(s)."""
    providers = [provider] if provider else list(Provider)
    ok = True

    for p in providers:
        ok = uninstall_from_provider(p, dry_run=dry_run, backup=backup) and ok

    if not ok:
        raise typer.Exit(1)


@app.command()
def status():
    """Show installation status for all providers."""
    table = Table(title="nbdev-mcp Status", show_header=True)
    table.add_column("Provider", style="cyan")
    table.add_column("Status")
    table.add_column("Format")
    table.add_column("Auto-Start")
    table.add_column("Path", style="dim")

    for provider in Provider:
        info = check_provider_status(provider)
        if not info["exists"]:
            st = "[dim]Config not found[/]"
            fmt = info.get("format", "")
            autostart = ""
        elif info["installed"]:
            st = "[green]✓ Installed[/]"
            fmt = info.get("format", "")
            if "autostart" in info:
                autostart = "[green]✓[/]" if info["autostart"] else "[yellow]✗[/]"
            else:
                autostart = "[dim]n/a[/]"
        else:
            st = "[yellow]Not configured[/]"
            fmt = info.get("format", "")
            autostart = ""

        table.add_row(provider.value.title(), st, fmt, autostart, info["path"])

    table.add_row("", "", "", "", "")
    table.add_row("Python", f"[green]v{sys.version.split()[0]}[/]", "", "", get_python_path())
    table.add_row("nbdev-mcp", f"[blue]v{__version__}[/]", "", "", "")

    console.print(table)


In [ ]:
#| export
@app.command()
def test(
    agent: Annotated[str, typer.Argument(
        help="Agent to test: claude or codex"
    )] = "claude",
    scenario: Annotated[Optional[str], typer.Option(
        "-s", "--scenario", help="Specific scenario to test (all if omitted)"
    )] = None,
    save_dir: Annotated[Optional[str], typer.Option(
        "-o", "--output", help="Directory to save session files"
    )] = None,
    timeout: Annotated[int, typer.Option(
        "-t", "--timeout", help="Timeout per test in seconds"
    )] = 120,
):
    """Run automated agent e2e tests.
    
    Tests agent behavior against expected patterns for MCP tool usage.
    """
    from nbdev_mcp.tests.agent_e2e import run_agent_test, run_all_agent_tests, SCENARIOS
    
    if scenario:
        if scenario not in SCENARIOS:
            console.print(f"[red]Unknown scenario: {scenario}[/]")
            console.print(f"Available: {', '.join(SCENARIOS.keys())}")
            raise typer.Exit(1)
        
        result = run_agent_test(
            agent=agent,
            scenario=scenario,
            timeout=timeout,
            save_session=Path(save_dir) / f"{scenario}.json" if save_dir else None
        )
        
        if result.get("ok"):
            console.print(f"[green]✓ PASS[/]: {scenario}")
        else:
            console.print(f"[red]✗ FAIL[/]: {scenario}")
            for failure in result.get("validation", {}).get("failures", []):
                console.print(f"  - {failure}")
    else:
        results = run_all_agent_tests(
            agent=agent,
            save_dir=Path(save_dir) if save_dir else None
        )
        
        console.print()
        table = Table(title=f"Agent Test Results ({agent})")
        table.add_column("Scenario")
        table.add_column("Status")
        table.add_column("Details")
        
        for r in results["results"]:
            status = "[green]PASS[/]" if r.get("ok") else "[red]FAIL[/]"
            details = ""
            if not r.get("ok"):
                failures = r.get("validation", {}).get("failures", [])
                details = failures[0] if failures else r.get("error", "")
            table.add_row(r.get("scenario", "?"), status, details[:50])
        
        console.print(table)
        console.print()
        console.print(f"Pass rate: {results['passed']}/{results['total']} ({results['pass_rate']:.0%})")

In [ ]:
#| export
def main():
    """Entry point for console script."""
    app()